In [1]:
%ls
import glob
import os
import cv2
import numpy as np


from driver import io_shape_dict
from driver_base import FINNExampleOverlay
from utils import letterbox, scale_coords
from driver import io_shape_dict
from driver_base import FINNExampleOverlay

coco_evaluate.py          finn/           qonnx/
coco_prep.py              finn-accel.bit  testing_on_fpga.ipynb
data/                     finn-accel.hwh  utils.py
driver_base.py            loaded.xclbin   validate.py
driver.py                 mul_0.npy
extract_output_scales.py  mul_1.npy


In [2]:
driver = FINNExampleOverlay(
    bitfile_name="./finn-accel.bit",
    platform="zynq-iodma",
    io_shape_dict=io_shape_dict,
    batch_size=1,
    runtime_weight_dir="runtime_weights/",
)

In [3]:
strides = [16, 32]
anchors_config = [
    [[10,14], [23,27], [37,58]],     # P4
    [[81,82], [135,169], [344,319]]  # P5
]
nc = 1
scale1 = np.load("./mul_0.npy")
scale2 = np.load("./mul_1.npy")
class_names = ['Person']

In [4]:
source = "/home/xilinx/jupyter_notebooks/pynq/data/inputs/"
output_folder = "test_outputs/"

In [6]:
!ls /home/xilinx/jupyter_notebooks/pynq/data/inputs/

0001_png_jpg.rf.192a313a2592b42b0d4019a32381c539.jpg
0001_png_jpg.rf.953215cc2fc0aac665a3afc395f9f727.jpg
010_jpg.rf.09200232cc364ae78190543d7ac54c8f.jpg
017-oslavanske_historicke_slavnosti-11_09_2010_jpg.rf.2248257f8b4f1fe39b0608acf0fdf8b1.jpg
0_193_jpg.rf.f9a09323a57269ad14bc750518ca94eb.jpg
090-oslavanske_historicke_slavnosti-11_09_2010_jpg.rf.d5eaef40c1bb815b21dc3a64761da433.jpg
100_1262_jpg.rf.fbbfd91a6a600c51053b5bc1e1806ce5.jpg
1_027_jpg.rf.fde2cd20c58add81a0f36c0e3e1cc38f.jpg
20100913_175_jpg.rf.c826fa46fa1cd54f9366c958d6fa955f.jpg
2016031508000001_jpg.rf.52d31c95cb5a60f6316938bae4a5d59e.jpg
2016031508000002_jpg.rf.1c3b8689ef74c99f4b7c50d254806e7a.jpg
2016031508000006_jpg.rf.a58d12cf6b092e3b8ad1d8edd0a7523e.jpg
2016031508000006_jpg.rf.fba0953e6b0a1b1d86acf8a39b0eb140.jpg
2016031508000007_jpg.rf.d112d466d18478e43487fae9df61eb05.jpg
2016031508000008_jpg.rf.a01d5ed30dd5441ab26e0672af968aff.jpg
2016031508000010_jpg.rf.d501aab88ee987b334a6f45689744c7a.jpg
2016031508000011_jpg.rf.a7b

img_0443_jpg.rf.db6b84544939b3dda07c53fa511cb88b.jpg
img_0444_jpg.rf.8427edcf885b8b01f783a190284ad1f4.jpg
img_0445_jpg.rf.3fbb89860108f67edf608fc85ffff84b.jpg
img_0446_jpg.rf.28f762222cf527c9a8b41bb3679bf892.jpg
img_0447_jpg.rf.e6b408ad9d6d271bcfe4e14cbf47cb6a.jpg
img_0449_jpg.rf.be676c9a1a556408b0e01bbfe0ab51e3.jpg
img_0450_jpg.rf.2c3ef1a7976a6851ee9b6a18c874cb62.jpg
img_0450_jpg.rf.d6356409e41eb086c523f5f25efceee4.jpg
img_0451_jpg.rf.50bffb3c9cf5f835b4884176bbc70bf2.jpg
img_0451_jpg.rf.eec10276b9e725a8fb477ff028e38249.jpg
img_0454_jpg.rf.3df418a9442b8fdb75993ff91e56cee3.jpg
img_0454_jpg.rf.8139c12b5981a57e5c3f74cf0eb03f24.jpg
img_0454_jpg.rf.86c72b0bfcbb37fd74cec876b6b485f0.jpg
img_0455_jpg.rf.ed27ae767794598cc7407fe7189ad2da.jpg
img_0455_jpg.rf.f37f5407fd18ae571a41c382bba320f8.jpg
img_0456_jpg.rf.f5755bceb4b6b5d7060f3f2c1a045ead.jpg
img_0457_jpg.rf.7fbd46d534f75fdc74354eb514f42f67.jpg
img_0458_jpg.rf.3eb7984471353d2c352c7f23980d0163.jpg
img_0458_jpg.rf.4c981428a993ac65ce839a919cb2b5

In [7]:
img_paths = glob.glob(source + "**/*.jpg", recursive=True)

In [8]:
len_img_paths = len(img_paths)
print(len_img_paths)

1879


In [9]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [10]:
def postprocess_yolo_outputs(outputs, anchors_config, strides, nc):
    """
    Post-process YOLO outputs from your FINN-exported model.

    Args:
        outputs: list of tensors [output_P4, output_P5]
                 shapes: [(1, 18, 26, 26), (1, 18, 13, 13)]
        anchors_config: list of anchor boxes per layer, [[na,2], [na,2]]
        strides: list of strides per layer, e.g. [16, 32]
        nc: number of classes

    Returns:
        predictions: [batch, num_predictions, nc+5] in format [x, y, w, h, obj, cls...]
    """
    z = []

    # Loop over layers
    for idx, output in enumerate(outputs):
        bs, _, ny, nx = output.shape  # e.g., (1, 18, 26, 26)
        anchors = np.array(anchors_config[idx], dtype=np.float32)  # [na, 2]
        stride = strides[idx]
        na = len(anchors)
        no = nc + 5

        # Reshape to [batch, na, no, ny, nx]
        output = output.reshape(bs, na, no, ny, nx)
        # Transpose to [batch, na, ny, nx, no]
        output = output.transpose(0, 1, 3, 4, 2)

        # Create grid
        yv, xv = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
        grid = np.stack((xv, yv), axis=2).reshape(1, 1, ny, nx, 2).astype(np.float32)

        # Create anchor grid
        anchor_grid = anchors.reshape(1, na, 1, 1, 2)

        # Sigmoid activation
        y_sigmoid = sigmoid(output)

        # xy coordinates (center) in absolute image scale
        xy = (y_sigmoid[..., 0:2] * 2.0 - 0.5 + grid) * stride
        # width/height in absolute scale
        wh = (y_sigmoid[..., 2:4] * 2.0) ** 2 * anchor_grid
        # objectness and class scores
        obj_cls = y_sigmoid[..., 4:]

        # Concatenate to [x_center, y_center, w, h, obj, cls...]
        y_out = np.concatenate([xy, wh, obj_cls], axis=-1)

        # Flatten to [batch, num_predictions, no]
        z.append(y_out.reshape(bs, -1, no))

    # Combine both layers
    return np.concatenate(z, axis=1)

In [11]:
def box_iou(box1, box2):
    """
    Compute IoU between two sets of boxes.
    Boxes are in [x1, y1, x2, y2] format.
    """
    # Expand dimensions for broadcasting
    box1 = np.expand_dims(box1, axis=1)  # [N, 1, 4]
    box2 = np.expand_dims(box2, axis=0)  # [1, M, 4]
    
    # Intersection coordinates
    inter_x1 = np.maximum(box1[..., 0], box2[..., 0])
    inter_y1 = np.maximum(box1[..., 1], box2[..., 1])
    inter_x2 = np.minimum(box1[..., 2], box2[..., 2])
    inter_y2 = np.minimum(box1[..., 3], box2[..., 3])
    
    # Intersection area
    inter_w = np.maximum(0, inter_x2 - inter_x1)
    inter_h = np.maximum(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    # Union area
    box1_area = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    box2_area = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    union_area = box1_area + box2_area - inter_area
    
    # IoU
    iou = inter_area / (union_area + 1e-6)
    return iou

In [12]:
def xywh2xyxy(boxes):
    """Convert boxes from [x_center, y_center, w, h] to [x1, y1, x2, y2]."""
    xy = boxes[..., :2]
    wh = boxes[..., 2:4]
    xy1 = xy - wh / 2
    xy2 = xy + wh / 2
    return np.concatenate([xy1, xy2], axis=-1)

In [13]:
def non_max_suppression(prediction, conf_thres=0.25, iou_thres=0.45):
    """
    Performs Non-Maximum Suppression (NMS) on inference results.
    Based on utils.general.non_max_suppression from your code.
    
    Returns:
         list of detections, on (n,6) tensor per image [xyxy, conf, cls]
    """
    xc = prediction[..., 4] > conf_thres  # candidates
    
    # Settings
    max_wh = 7680  # maximum box width and height
    max_nms = 30000  # maximum number of boxes into torchvision.ops.nms()
    
    output = []
    for xi, x in enumerate(prediction):  # image index, image inference
        # Apply constraints
        x = x[xc[xi]]  # confidence
        
        # If none remain process next image
        if not x.shape[0]:
            output.append(np.zeros((0, 6)))
            continue
        
        # Compute conf
        x[:, 5:] *= x[:, 4:5]  # conf = obj_conf * cls_conf
        
        # Box (center x, center y, width, height) to (x1, y1, x2, y2)
        box = xywh2xyxy(x[:, :4])
        
        # Detections matrix nx6 (xyxy, conf, cls)
        conf = np.max(x[:, 5:], axis=1, keepdims=True)
        j = np.argmax(x[:, 5:], axis=1).reshape(-1, 1)
        x = np.concatenate((box, conf, j.astype(np.float32)), axis=1)[conf.reshape(-1) > conf_thres]
        
        # Check shape
        n = x.shape[0]  # number of boxes
        if not n:
            output.append(np.zeros((0, 6)))
            continue
        
        # Sort by confidence
        x = x[x[:, 4].argsort()[::-1][:max_nms]]
        
        # Batched NMS
        c = x[:, 5:6] * max_wh  # classes
        boxes, scores = x[:, :4] + c, x[:, 4]  # boxes (offset by class), scores
        
        # NMS
        keep = []
        indices = np.arange(x.shape[0])
        
        while boxes.shape[0]:
            # Pick detection with highest confidence (already sorted, so index 0)
            keep.append(indices[0])
            
            if boxes.shape[0] == 1:
                break
            
            # Compute IoU with remaining boxes
            ious = box_iou(boxes[0:1], boxes[1:])
            
            # Remove boxes with high IoU
            mask = ious[0] < iou_thres
            boxes = boxes[1:][mask]
            scores = scores[1:][mask]
            indices = indices[1:][mask]
        
        if keep:
            keep_indices = np.array(keep)
            output.append(x[keep_indices])
        else:
            output.append(np.zeros((0, 6)))
    
    return output

In [14]:
def draw_detections(image, detections, class_names=None):
    """Draw bounding boxes on image."""
    image_draw = image.copy()
    
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cls = int(cls)
        
        # Draw box
        color = (0, 255, 0)
        cv2.rectangle(image_draw, (x1, y1), (x2, y2), color, 2)
        
        # Draw label
        label = f"{class_names[cls] if class_names else cls}: {conf:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(image_draw, (x1, y1 - label_h - 10), (x1 + label_w, y1), color, -1)
        cv2.putText(image_draw, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    
    return image_draw

In [17]:
img_size = (416, 416)
conf_thres = 0.001
iou_thres = 0.45

for number, test_img_path in enumerate(img_paths):
    print(f"[INFO] processing {number+1}/{len_img_paths}")
    img_org = cv2.imread(test_img_path)
    img = img_org.copy()

    h, w, _ = img_org.shape
    img, ratio, (dw, dh) = letterbox(img, img_size, auto=False)

    img = img[:, :, ::-1]
    img = img.astype(np.uint8)

    driver_in = np.expand_dims(img, 0)

    output = driver.execute(driver_in)
    output1 = output[0].transpose(0, 3, 1, 2) 
    output2 = output[1].transpose(0, 3, 1, 2) 
    
    # Scaling
    output1 = output1 * scale1
    output2 = output2 * scale2
    
    outputs = [output2, output1]
    
    # Post-processing
    pred = postprocess_yolo_outputs(
        outputs,
        anchors_config=anchors_config,
        strides=strides,
        nc=nc
    )
    
    print(f"Prediction shape: {pred.shape}")
    print(f"Number of anchor prediction: {pred.shape[1]}")
    
    # Apply NMS
    print("Applying NMS...")
    print(f"Number of predictions > {conf_thres}: {(pred[..., 4] > conf_thres).sum()}")
    
    pred = non_max_suppression(pred, conf_thres, iou_thres)
    
    # Get detected boxes_detected, labels, confidences, class-scores.
    
    boxes_detected, class_names_detected, probs_detected = [], [], []

    folder_name, file_name = test_img_path.split("/")[-2:]
    folder_name = folder_name.replace('images', 'labels') 
    save_folder = output_folder + folder_name
    
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    save_path = save_folder + "/" + file_name.replace(".jpg", ".txt")
    f = open(save_path, "w")
    det = pred[0] if len(pred)>0 else []

    # Process predictions
    if len(det):
        f.write(file_name+"\n")
        f.write(str(len(det))+"\n")
        # Rescale boxes from img_size to im0 size
        det[:, :4] = scale_coords(img.shape, det[:, :4], img_org.shape).round() # type: ignore
        
        # Draw and save result
        result_image = draw_detections(img, det, class_names)
        output_path = f'output_detection_{number+1}.jpg'
        cv2.imwrite(output_path, result_image)
        print(f"\nResult saved to {output_path}")
        
        for i, d in enumerate(det):
            x1, y1, x2, y2, conf, cls = d
            class_name = class_names[int(cls)] if class_names else int(cls)
            print(f"  Detection {i+1}: {class_name} ({conf:.3f}) at [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]")
            f.write("%.5f %.5f %.5f %.5f %g\n"%(d[0], d[1], d[2]-d[0], d[3]-d[1], d[4]))
    f.close()
    

[INFO] processing 1/1879
Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 299

Result saved to output_detection_1.jpg
  Detection 1: Person (0.429) at [214.0, 114.0, 283.0, 212.0]
  Detection 2: Person (0.426) at [104.0, 124.0, 208.0, 234.0]
  Detection 3: Person (0.408) at [190.0, 52.0, 230.0, 130.0]
  Detection 4: Person (0.373) at [239.0, 84.0, 265.0, 122.0]
  Detection 5: Person (0.372) at [280.0, 91.0, 308.0, 136.0]
  Detection 6: Person (0.350) at [123.0, 91.0, 174.0, 176.0]
  Detection 7: Person (0.347) at [267.0, 109.0, 328.0, 220.0]
  Detection 8: Person (0.178) at [277.0, 98.0, 314.0, 158.0]
  Detection 9: Person (0.168) at [206.0, 72.0, 245.0, 122.0]
  Detection 10: Person (0.163) at [257.0, 84.0, 300.0, 124.0]
  Detection 11: Person (0.123) at [242.0, 109.0, 304.0, 197.0]
  Detection 12: Person (0.118) at [160.0, 81.0, 212.0, 146.0]
  Detection 13: Person (0.030) at [226.0, 79.0, 259.0, 130.0]
  Detection 14: Pe

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 241

Result saved to output_detection_9.jpg
  Detection 1: Person (0.240) at [97.0, 30.0, 107.0, 64.0]
  Detection 2: Person (0.204) at [77.0, 373.0, 167.0, 416.0]
  Detection 3: Person (0.076) at [311.0, 15.0, 320.0, 43.0]
  Detection 4: Person (0.036) at [157.0, 9.0, 166.0, 42.0]
  Detection 5: Person (0.029) at [87.0, 32.0, 98.0, 66.0]
  Detection 6: Person (0.026) at [105.0, 379.0, 203.0, 416.0]
  Detection 7: Person (0.021) at [102.0, 23.0, 112.0, 60.0]
  Detection 8: Person (0.014) at [81.0, 389.0, 142.0, 416.0]
  Detection 9: Person (0.011) at [14.0, 68.0, 28.0, 105.0]
  Detection 10: Person (0.011) at [6.0, 80.0, 21.0, 120.0]
  Detection 11: Person (0.011) at [0.0, 175.0, 23.0, 352.0]
  Detection 12: Person (0.010) at [318.0, 18.0, 327.0, 46.0]
  Detection 13: Person (0.009) at [50.0, 339.0, 182.0, 416.0]
  Detection 14: Person (0.008) at [0.0, 118.0, 18.0, 291.0]
  D

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 226

Result saved to output_detection_12.jpg
  Detection 1: Person (0.349) at [169.0, 4.0, 184.0, 46.0]
  Detection 2: Person (0.333) at [351.0, 45.0, 363.0, 86.0]
  Detection 3: Person (0.256) at [95.0, 15.0, 104.0, 50.0]
  Detection 4: Person (0.215) at [164.0, 1.0, 179.0, 40.0]
  Detection 5: Person (0.195) at [23.0, 50.0, 34.0, 90.0]
  Detection 6: Person (0.130) at [304.0, 16.0, 315.0, 51.0]
  Detection 7: Person (0.076) at [359.0, 45.0, 370.0, 86.0]
  Detection 8: Person (0.074) at [89.0, 14.0, 101.0, 48.0]
  Detection 9: Person (0.071) at [221.0, 0.0, 232.0, 33.0]
  Detection 10: Person (0.065) at [17.0, 57.0, 31.0, 93.0]
  Detection 11: Person (0.063) at [226.0, 1.0, 237.0, 23.0]
  Detection 12: Person (0.060) at [14.0, 57.0, 24.0, 94.0]
  Detection 13: Person (0.034) at [158.0, 5.0, 168.0, 39.0]
  Detection 14: Person (0.028) at [217.0, 4.0, 227.0, 29.0]
  Detection 

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 89

Result saved to output_detection_18.jpg
  Detection 1: Person (0.415) at [100.0, 107.0, 147.0, 181.0]
  Detection 2: Person (0.386) at [138.0, 68.0, 167.0, 123.0]
  Detection 3: Person (0.004) at [134.0, 87.0, 162.0, 130.0]
  Detection 4: Person (0.003) at [94.0, 56.0, 126.0, 116.0]
  Detection 5: Person (0.002) at [106.0, 67.0, 141.0, 116.0]
[INFO] processing 19/1879
Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 78

Result saved to output_detection_19.jpg
  Detection 1: Person (0.432) at [95.0, 102.0, 148.0, 179.0]
  Detection 2: Person (0.359) at [145.0, 60.0, 173.0, 115.0]
  Detection 3: Person (0.004) at [104.0, 57.0, 139.0, 108.0]
  Detection 4: Person (0.003) at [84.0, 49.0, 118.0, 111.0]
  Detection 5: Person (0.002) at [96.0, 45.0, 127.0, 106.0]
[INFO] processing 20/1879
Prediction shape: (1, 2535, 

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 171

Result saved to output_detection_23.jpg
  Detection 1: Person (0.143) at [66.0, 34.0, 80.0, 60.0]
  Detection 2: Person (0.138) at [277.0, 3.0, 292.0, 36.0]
  Detection 3: Person (0.040) at [41.0, 49.0, 55.0, 82.0]
  Detection 4: Person (0.032) at [222.0, 1.0, 233.0, 26.0]
  Detection 5: Person (0.024) at [47.0, 40.0, 61.0, 73.0]
  Detection 6: Person (0.023) at [215.0, 1.0, 226.0, 24.0]
  Detection 7: Person (0.022) at [60.0, 32.0, 72.0, 61.0]
  Detection 8: Person (0.016) at [282.0, 1.0, 299.0, 30.0]
  Detection 9: Person (0.015) at [138.0, 4.0, 151.0, 36.0]
  Detection 10: Person (0.014) at [270.0, 4.0, 283.0, 33.0]
  Detection 11: Person (0.011) at [16.0, 59.0, 33.0, 98.0]
  Detection 12: Person (0.010) at [398.0, 118.0, 416.0, 275.0]
  Detection 13: Person (0.009) at [384.0, 94.0, 415.0, 258.0]
  Detection 14: Person (0.007) at [359.0, 24.0, 378.0, 74.0]
  Detection

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 456

Result saved to output_detection_25.jpg
  Detection 1: Person (0.370) at [308.0, 85.0, 368.0, 242.0]
  Detection 2: Person (0.357) at [104.0, 114.0, 204.0, 349.0]
  Detection 3: Person (0.185) at [158.0, 60.0, 203.0, 200.0]
  Detection 4: Person (0.140) at [70.0, 207.0, 213.0, 396.0]
  Detection 5: Person (0.113) at [318.0, 317.0, 407.0, 416.0]
  Detection 6: Person (0.072) at [142.0, 53.0, 208.0, 282.0]
  Detection 7: Person (0.045) at [154.0, 70.0, 191.0, 151.0]
  Detection 8: Person (0.031) at [364.0, 262.0, 416.0, 413.0]
  Detection 9: Person (0.025) at [0.0, 324.0, 48.0, 415.0]
  Detection 10: Person (0.013) at [294.0, 309.0, 373.0, 410.0]
  Detection 11: Person (0.008) at [360.0, 339.0, 415.0, 416.0]
  Detection 12: Person (0.006) at [46.0, 365.0, 185.0, 415.0]
  Detection 13: Person (0.004) at [304.0, 356.0, 384.0, 415.0]
  Detection 14: Person (0.004) at [0.0, 27

... (441 outputs left out)

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 156

Result saved to output_detection_1857.jpg
  Detection 1: Person (0.089) at [87.0, 38.0, 98.0, 72.0]
  Detection 2: Person (0.088) at [93.0, 36.0, 102.0, 72.0]
  Detection 3: Person (0.077) at [301.0, 15.0, 312.0, 41.0]
  Detection 4: Person (0.039) at [12.0, 79.0, 25.0, 121.0]
  Detection 5: Person (0.038) at [225.0, 3.0, 235.0, 35.0]
  Detection 6: Person (0.030) at [49.0, 52.0, 62.0, 93.0]
  Detection 7: Person (0.021) at [8.0, 88.0, 23.0, 131.0]
  Detection 8: Person (0.019) at [35.0, 64.0, 48.0, 101.0]
  Detection 9: Person (0.017) at [44.0, 61.0, 57.0, 99.0]
  Detection 10: Person (0.013) at [63.0, 50.0, 75.0, 84.0]
  Detection 11: Person (0.012) at [4.0, 92.0, 17.0, 131.0]
  Detection 12: Person (0.010) at [75.0, 41.0, 88.0, 78.0]
  Detection 13: Person (0.007) at [404.0, 95.0, 416.0, 178.0]
  Detection 14: Person (0.006) at [17.0, 72.0, 32.0, 108.0]
  Detection 15

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 111

Result saved to output_detection_1863.jpg
  Detection 1: Person (0.406) at [257.0, 44.0, 280.0, 114.0]
  Detection 2: Person (0.385) at [167.0, 45.0, 203.0, 120.0]
  Detection 3: Person (0.007) at [111.0, 44.0, 127.0, 97.0]
  Detection 4: Person (0.005) at [155.0, 53.0, 193.0, 134.0]
  Detection 5: Person (0.003) at [184.0, 39.0, 210.0, 90.0]
  Detection 6: Person (0.002) at [163.0, 52.0, 190.0, 99.0]
  Detection 7: Person (0.002) at [178.0, 29.0, 201.0, 60.0]
  Detection 8: Person (0.002) at [175.0, 41.0, 201.0, 81.0]
  Detection 9: Person (0.002) at [383.0, 217.0, 416.0, 377.0]
  Detection 10: Person (0.002) at [269.0, 51.0, 300.0, 127.0]
  Detection 11: Person (0.001) at [108.0, 54.0, 133.0, 129.0]
  Detection 12: Person (0.001) at [355.0, 224.0, 401.0, 381.0]
[INFO] processing 1864/1879
Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...


Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 97

Result saved to output_detection_1871.jpg
  Detection 1: Person (0.419) at [279.0, 116.0, 318.0, 188.0]
  Detection 2: Person (0.367) at [288.0, 59.0, 319.0, 118.0]
  Detection 3: Person (0.029) at [269.0, 69.0, 302.0, 127.0]
  Detection 4: Person (0.007) at [231.0, 72.0, 268.0, 129.0]
  Detection 5: Person (0.004) at [224.0, 63.0, 258.0, 117.0]
  Detection 6: Person (0.004) at [292.0, 87.0, 326.0, 141.0]
  Detection 7: Person (0.002) at [196.0, 58.0, 227.0, 112.0]
  Detection 8: Person (0.002) at [289.0, 109.0, 322.0, 160.0]
  Detection 9: Person (0.001) at [401.0, 217.0, 416.0, 339.0]
  Detection 10: Person (0.001) at [304.0, 62.0, 333.0, 122.0]
[INFO] processing 1872/1879
Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 78

Result saved to output_detection_1872.jpg
  Detection 1: Person (0.231) at [235.0, 7

Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 138

Result saved to output_detection_1877.jpg
  Detection 1: Person (0.439) at [141.0, 106.0, 172.0, 159.0]
  Detection 2: Person (0.422) at [214.0, 86.0, 244.0, 130.0]
  Detection 3: Person (0.410) at [191.0, 83.0, 213.0, 120.0]
  Detection 4: Person (0.120) at [154.0, 99.0, 187.0, 147.0]
  Detection 5: Person (0.023) at [55.0, 99.0, 71.0, 164.0]
  Detection 6: Person (0.008) at [196.0, 85.0, 230.0, 124.0]
  Detection 7: Person (0.002) at [40.0, 92.0, 54.0, 158.0]
  Detection 8: Person (0.002) at [367.0, 357.0, 416.0, 416.0]
  Detection 9: Person (0.001) at [329.0, 251.0, 412.0, 416.0]
[INFO] processing 1878/1879
Prediction shape: (1, 2535, 6)
Number of anchor prediction: 2535
Applying NMS...
Number of predictions > 0.001: 104

Result saved to output_detection_1878.jpg
  Detection 1: Person (0.444) at [99.0, 109.0, 149.0, 180.0]
  Detection 2: Person (0.424) at [74.0, 141.0